# Proyek Analisis Data: Bike Sharing Dataset
- **Nama:** [Input Nama]
- **Email:** [Input Email]
- **ID Dicoding:** [Input Username]

## Menentukan Pertanyaan Bisnis

- **Pertanyaan 1:** Faktor cuaca apa saja (suhu, kelembapan, kecepatan angin) yang paling memengaruhi jumlah penyewaan sepeda harian pada tahun 2011–2012?
- **Pertanyaan 2:** Bagaimana pola penyewaan sepeda berdasarkan hari kerja vs hari libur dan musim, selama tahun 2011–2012?

## Import Semua Packages/Library yang Digunakan

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('Library berhasil diimport!')

## Data Wrangling

### Gathering Data

#### Load Dataset Bike Sharing (day.csv)

In [ ]:
day_df = pd.read_csv('data/day.csv')

print('Shape dataset:', day_df.shape)
print('5 baris pertama:')
day_df.head()

**Insight:**
- Dataset terdiri dari 731 baris dan 16 kolom, mencakup data penyewaan sepeda harian selama 2 tahun (2011–2012).
- Kolom utama yang akan dianalisis: `cnt` (total penyewaan), `temp`, `hum`, `windspeed`, `season`, `workingday`, dan `weathersit`.

### Assessing Data

#### Identifying missing value, duplicate, dan tipe data

In [ ]:
# Cek tipe data dan missing value
print('=== Info Dataset ===')
day_df.info()
print()

# Cek missing value
print('=== Missing Values ===')
print(day_df.isnull().sum())
print()

# Cek duplikat
print('=== Duplikat ===')
print(f'Jumlah baris duplikat: {day_df.duplicated().sum()}')

**Steps to Take:**
- Tidak ditemukan missing value → tidak perlu penanganan khusus.
- Tidak ditemukan baris duplikat → data sudah bersih dari duplikasi.
- Kolom `dteday` masih bertipe object, perlu dikonversi ke datetime.

**Insight:**
- Dataset memiliki kualitas yang baik: tidak ada missing value maupun duplikat.
- Kolom numerik seperti `temp`, `hum`, `windspeed` sudah dalam format normalized (0–1).

#### Identifying invalid value dan inkonsistensi data

In [ ]:
# Cek nilai tidak wajar
print('=== Cek Nilai di Luar Batas Normal ===')
for col in ['temp', 'atemp', 'hum', 'windspeed']:
    out = ((day_df[col] < 0) | (day_df[col] > 1)).sum()
    print(f"Nilai di luar [0,1] pada '{col}': {out}")

print()
# Cek konsistensi cnt = casual + registered
inconsistent = (day_df['cnt'] != day_df['casual'] + day_df['registered']).sum()
print(f'Inkonsistensi cnt vs casual+registered: {inconsistent}')

print()
# Statistik deskriptif
print('=== Statistik Deskriptif ===')
day_df[['temp','hum','windspeed','cnt']].describe()

**Steps to Take:**
- Tidak ditemukan nilai di luar batas normal pada kolom cuaca.
- Nilai `cnt` konsisten dengan `casual + registered`.
- Data siap dilanjutkan ke tahap Cleaning.

### Cleaning Data

#### Fixing tipe data dan menambahkan kolom label untuk analisis

In [ ]:
# Konversi tipe data dteday ke datetime
day_df['dteday'] = pd.to_datetime(day_df['dteday'])

# Mapping label kategorikal
season_map = {1: 'Spring', 2: 'Summer', 3: 'Fall', 4: 'Winter'}
weathersit_map = {1: 'Clear', 2: 'Cloudy', 3: 'Light Rain/Snow', 4: 'Heavy Rain/Snow'}
workingday_map = {0: 'Holiday/Weekend', 1: 'Working Day'}
yr_map = {0: '2011', 1: '2012'}

day_df['season_label'] = day_df['season'].map(season_map)
day_df['weathersit_label'] = day_df['weathersit'].map(weathersit_map)
day_df['workingday_label'] = day_df['workingday'].map(workingday_map)
day_df['yr_label'] = day_df['yr'].map(yr_map)

# Denormalisasi nilai cuaca agar mudah dibaca
day_df['temp_celsius'] = day_df['temp'] * 41        # max temp 41°C
day_df['hum_pct'] = day_df['hum'] * 100             # dalam persen
day_df['windspeed_kph'] = day_df['windspeed'] * 67  # max 67 km/h

print('Cleaning selesai!')
print('Shape setelah cleaning:', day_df.shape)
day_df.head()

**Insight:**
- Tipe data `dteday` sudah dikonversi ke datetime untuk memudahkan analisis tren waktu.
- Kolom label kategorikal ditambahkan agar visualisasi lebih mudah dipahami.
- Nilai suhu, kelembapan, dan kecepatan angin didenormalisasi ke satuan aslinya.

## Exploratory Data Analysis (EDA)

### Explore pengaruh faktor cuaca terhadap jumlah penyewaan sepeda

In [ ]:
# Korelasi variabel cuaca dengan cnt
print('=== Korelasi Variabel Cuaca dengan Jumlah Penyewaan ===')
cuaca_cols = ['temp_celsius', 'hum_pct', 'windspeed_kph', 'cnt']
corr = day_df[cuaca_cols].corr()
print(corr['cnt'].drop('cnt').sort_values(ascending=False))

print()
# Rata-rata penyewaan per kondisi cuaca
print('=== Rata-rata Penyewaan per Kondisi Cuaca ===')
print(day_df.groupby('weathersit_label')['cnt'].agg(['mean','median','count']).sort_values('mean', ascending=False))

**Insight:**
- Suhu (`temp_celsius`) memiliki korelasi positif paling kuat terhadap jumlah penyewaan.
- Kelembapan (`hum_pct`) berkorelasi negatif — semakin lembap, semakin sedikit penyewaan.
- Kecepatan angin berkorelasi negatif lemah.
- Kondisi cuaca cerah (Clear) menghasilkan rata-rata penyewaan tertinggi.

### Explore pola penyewaan berdasarkan musim dan hari kerja

In [ ]:
# Rata-rata penyewaan per musim
print('=== Rata-rata Penyewaan per Musim ===')
print(day_df.groupby('season_label')['cnt'].agg(['mean','median','count']).sort_values('mean', ascending=False))

print()
# Rata-rata penyewaan hari kerja vs libur
print('=== Rata-rata Penyewaan: Hari Kerja vs Libur ===')
print(day_df.groupby('workingday_label')['cnt'].agg(['mean','median','count']))

print()
# Pivot: musim x jenis hari
print('=== Rata-rata Penyewaan per Musim & Jenis Hari ===')
pivot = day_df.pivot_table(values='cnt', index='season_label', columns='workingday_label', aggfunc='mean')
print(pivot.round(0))

**Insight:**
- Musim Fall (Gugur) mencatat rata-rata penyewaan tertinggi, diikuti Summer, Winter, dan Spring.
- Hari kerja memiliki rata-rata penyewaan lebih tinggi dibanding hari libur/akhir pekan.
- Perbedaan paling mencolok terjadi pada musim Fall dan Summer.

## Visualization & Explanatory Analysis

### Pertanyaan 1: Faktor cuaca apa saja yang paling memengaruhi jumlah penyewaan sepeda harian pada tahun 2011–2012?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Pengaruh Faktor Cuaca terhadap Jumlah Penyewaan Sepeda (2011-2012)',
             fontsize=13, fontweight='bold', y=1.02)

cuaca_vars = [
    ('temp_celsius', 'Suhu (°C)', '#e74c3c'),
    ('hum_pct', 'Kelembapan (%)', '#3498db'),
    ('windspeed_kph', 'Kecepatan Angin (km/h)', '#2ecc71')
]

for ax, (col, label, color) in zip(axes, cuaca_vars):
    ax.scatter(day_df[col], day_df['cnt'], alpha=0.4, color=color, s=20)
    z = np.polyfit(day_df[col], day_df['cnt'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(day_df[col].min(), day_df[col].max(), 100)
    ax.plot(x_line, p(x_line), color='black', linewidth=2, linestyle='--', label='Tren')
    corr_val = day_df[col].corr(day_df['cnt'])
    ax.set_title(f'{label}\nr = {corr_val:.2f}', fontsize=11)
    ax.set_xlabel(label)
    ax.set_ylabel('Jumlah Penyewaan')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('visualization/plot_cuaca.png', dpi=150, bbox_inches='tight')
plt.show()

### Pertanyaan 2: Bagaimana pola penyewaan sepeda berdasarkan hari kerja vs hari libur dan musim, selama tahun 2011–2012?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Pola Penyewaan Sepeda Berdasarkan Musim & Jenis Hari (2011-2012)',
             fontsize=13, fontweight='bold')

# Plot 1 — Per musim
season_order = ['Spring', 'Summer', 'Fall', 'Winter']
season_avg = day_df.groupby('season_label')['cnt'].mean().reindex(season_order)
colors_season = ['#2ecc71', '#f39c12', '#e74c3c', '#3498db']
bars = axes[0].bar(season_avg.index, season_avg.values, color=colors_season,
                   edgecolor='white', linewidth=1.5)
axes[0].set_title('Rata-rata Penyewaan per Musim', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Musim')
axes[0].set_ylabel('Rata-rata Penyewaan')
axes[0].grid(axis='y', alpha=0.3)
for bar, val in zip(bars, season_avg.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
                 f'{val:.0f}', ha='center', fontsize=10, fontweight='bold')

# Plot 2 — Hari kerja vs libur per musim
pivot = day_df.pivot_table(values='cnt', index='season_label',
                            columns='workingday_label', aggfunc='mean').reindex(season_order)
x = np.arange(len(pivot.index))
width = 0.35
axes[1].bar(x - width/2, pivot['Holiday/Weekend'], width,
            label='Holiday/Weekend', color='#e74c3c', alpha=0.85)
axes[1].bar(x + width/2, pivot['Working Day'], width,
            label='Working Day', color='#3498db', alpha=0.85)
axes[1].set_title('Penyewaan: Hari Kerja vs Libur per Musim', fontsize=12, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(pivot.index)
axes[1].set_ylabel('Rata-rata Penyewaan')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('visualization/plot_pola.png', dpi=150, bbox_inches='tight')
plt.show()

**Insight:**
- Musim Fall menghasilkan penyewaan tertinggi karena cuaca paling nyaman untuk bersepeda.
- Hari kerja secara konsisten memiliki penyewaan lebih tinggi di semua musim → mayoritas pengguna adalah komuter.
- Musim Spring memiliki penyewaan terendah, dengan gap terbesar antara hari kerja dan libur.

## Conclusion & Recommendation

- **Conclusion pertanyaan 1:** Suhu adalah faktor cuaca yang paling kuat memengaruhi jumlah penyewaan sepeda (korelasi positif, r ≈ 0.63). Kelembapan berkorelasi negatif sedang (r ≈ -0.32), sementara kecepatan angin berkorelasi negatif lemah (r ≈ -0.23). Kondisi cuaca cerah juga menghasilkan rata-rata penyewaan jauh lebih tinggi dibanding cuaca buruk.
- **Conclusion pertanyaan 2:** Penyewaan sepeda tertinggi terjadi pada musim Fall, diikuti Summer, Winter, dan Spring. Hari kerja secara konsisten memiliki penyewaan lebih tinggi dibanding hari libur/akhir pekan di semua musim, mengindikasikan bahwa mayoritas pengguna memanfaatkan layanan ini sebagai transportasi komuter harian.

**Rekomendasi Action Item:**
- Tingkatkan ketersediaan unit sepeda dan promosi pada hari-hari bersuhu hangat (>20°C) dan kelembapan rendah untuk memaksimalkan penyewaan.
- Fokuskan program diskon dan promosi rekreasional pada musim Spring dan hari akhir pekan, karena segmen ini masih rendah dan berpotensi ditingkatkan.
- Pastikan ketersediaan armada yang cukup pada hari kerja di musim Fall dan Summer, karena kedua kondisi tersebut secara konsisten menghasilkan penyewaan tertinggi.